In [ ]:
# BAGGING CLASSIFIER MODEL(RUN IN COLAB ONLY)

In [4]:

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score


In [5]:
# first upload csv in colab
from google.colab import files
uploaded = files.upload()


Saving smoking.csv to smoking.csv


In [7]:
# then view csv
df = pd.read_csv('smoking.csv')
df.head()

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking
0,0,F,40,155,60,81.3,1.2,1.0,1.0,1.0,...,12.9,1.0,0.7,18.0,19.0,27.0,Y,0,Y,0
1,1,F,40,160,60,81.0,0.8,0.6,1.0,1.0,...,12.7,1.0,0.6,22.0,19.0,18.0,Y,0,Y,0
2,2,M,55,170,60,80.0,0.8,0.8,1.0,1.0,...,15.8,1.0,1.0,21.0,16.0,22.0,Y,0,N,1
3,3,M,40,165,70,88.0,1.5,1.5,1.0,1.0,...,14.7,1.0,1.0,19.0,26.0,18.0,Y,0,Y,0
4,4,F,40,155,60,86.0,1.0,1.0,1.0,1.0,...,12.5,1.0,0.6,16.0,14.0,22.0,Y,0,N,0


In [8]:
X = df[['gender','height(cm)','Gtp','hemoglobin','triglyceride','age','weight(kg)',
        'waist(cm)','HDL','serum creatinine','ALT','fasting blood sugar','relaxation',
        'LDL','systolic']]
y = df['smoking']

In [9]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['tartar'] = le.fit_transform(df['tartar'])
df['dental caries'] = le.fit_transform(df['dental caries'])


In [15]:
 # convert string to float
import pandas as pd
x_train = pd.get_dummies(x_train, drop_first=True)
x_test = pd.get_dummies(x_test, drop_first=True)


In [16]:
model= BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100)
model.fit(x_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100)

In [17]:
y_pred = model.predict(x_test)


In [18]:

accuracy=accuracy_score(y_test, y_pred)
clf_report=classification_report(y_test, y_pred)

print("Accuracy:",accuracy )
print("classification report:\n",clf_report)


Accuracy: 0.8252985007630846
classification report:
               precision    recall  f1-score   support

           0       0.87      0.85      0.86      7027
           1       0.75      0.79      0.77      4112

    accuracy                           0.83     11139
   macro avg       0.81      0.82      0.81     11139
weighted avg       0.83      0.83      0.83     11139



## Deployment addition (minimal; original cells preserved)
This section is appended only to save the trained model for deployment. The original notebook cells above are preserved.

In [ ]:
# Minimal deployment addition: save a final Random Forest + preprocessing in model.pkl
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

DEPLOY_FEATURES = ['gender','height(cm)','Gtp','hemoglobin','triglyceride','age','weight(kg)',
                   'waist(cm)','HDL','serum creatinine','ALT','fasting blood sugar','relaxation',
                   'LDL','systolic']
X_deploy = df[DEPLOY_FEATURES].copy()
# The original notebook already label-encodes gender; this mapping keeps deployment consistent.
if X_deploy['gender'].dtype == 'object':
    X_deploy['gender'] = X_deploy['gender'].map({'F': 0, 'M': 1})
X_deploy = X_deploy.astype(float)
y_deploy = df['smoking']
deploy_scaler = StandardScaler()
X_deploy_scaled = deploy_scaler.fit_transform(X_deploy)
deploy_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
deploy_model.fit(X_deploy_scaled, y_deploy)
model_bundle = {'model': deploy_model, 'scaler': deploy_scaler, 'features': DEPLOY_FEATURES,
                'gender_mapping': {'F': 0, 'M': 1}, 'target': 'smoking'}
with open('model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print('model.pkl saved successfully.')
